In [1]:
# ============================================================
# NB01_Bronze_Ingestion
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

from datetime import datetime, timezone
from uuid import uuid4

# ============================================================
# 1. CONFIGURATION
# ============================================================

LakehouseName = "ProductCatalogLakehouse"

BronzeBasePath = "Files/bronze"

SourceConfigurations = {

    "StyleHaus": {
        "FilePath": f"{BronzeBasePath}/stylehaus/stylehaus_products.csv",
        "TableName": "bronze.StyleHausProducts",
        "ExpectedRecordCount": 100
    },

    "UrbanThread": {
        "FilePath": f"{BronzeBasePath}/urbanthread/urbanthread_products.csv",
        "TableName": "bronze.UrbanThreadProducts",
        "ExpectedRecordCount": 100
    }
}


StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 3, Finished, Available, Finished, False)

In [2]:
# ============================================================
# 2. INGESTION METADATA
# ============================================================

BatchID = str(uuid4())

IngestionTimestamp = datetime.now(timezone.utc)

print("BRONZE INGESTION CONFIGURATION")
print("=" * 60)

print(f"Lakehouse          : {LakehouseName}")
print(f"Bronze Base Path   : {BronzeBasePath}")
print(f"Batch ID           : {BatchID}")
print(f"Ingestion Timestamp: {IngestionTimestamp}")

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 4, Finished, Available, Finished, False)

BRONZE INGESTION CONFIGURATION
Lakehouse          : ProductCatalogLakehouse
Bronze Base Path   : Files/bronze
Batch ID           : a079f8b0-7e6e-45c9-af63-f34a1698ca8b
Ingestion Timestamp: 2026-11-03 07:48:39.877630+00:00


In [3]:
# ============================================================
# 3. VALIDATE SOURCE FILE AVAILABILITY
# ============================================================

print("\nSOURCE FILE VALIDATION")
print("=" * 60)

for SourceName, Config in SourceConfigurations.items():

    FilePath = Config["FilePath"]

    Files = notebookutils.fs.ls(FilePath.rsplit("/", 1)[0])

    MatchingFiles = [
        File for File in Files
        if File.name == FilePath.rsplit("/", 1)[1]
    ]

    if not MatchingFiles:
        raise FileNotFoundError(
            f"{SourceName} source file not found: {FilePath}"
        )

    if MatchingFiles[0].size == 0:
        raise ValueError(
            f"{SourceName} source file is empty: {FilePath}"
        )

    print(
        f"SUCCESS | {SourceName} | "
        f"{MatchingFiles[0].size} bytes"
    )

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 5, Finished, Available, Finished, False)


SOURCE FILE VALIDATION
SUCCESS | StyleHaus | 21002 bytes
SUCCESS | UrbanThread | 13271 bytes


In [4]:
# ============================================================
# 4. READ STYLEHAUS CSV
# ============================================================

StyleHausDf = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("mode", "FAILFAST")
    .load(SourceConfigurations["StyleHaus"]["FilePath"])
)


StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 6, Finished, Available, Finished, False)

In [5]:
# ============================================================
# 5. READ URBANTHREAD CSV
# ============================================================

UrbanThreadDf = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("mode", "FAILFAST")
    .load(SourceConfigurations["UrbanThread"]["FilePath"])
)


StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 7, Finished, Available, Finished, False)

In [6]:
# ============================================================
# 6. VALIDATE SOURCE DATA
# ============================================================

StyleHausCount = StyleHausDf.count()

UrbanThreadCount = UrbanThreadDf.count()

StyleHausColumnCount = len(StyleHausDf.columns)

UrbanThreadColumnCount = len(UrbanThreadDf.columns)


print("\nSOURCE DATA VALIDATION")
print("=" * 60)

print("\nSTYLEHAUS")
print("-" * 40)

print(f"Expected Records : {SourceConfigurations['StyleHaus']['ExpectedRecordCount']}")
print(f"Actual Records   : {StyleHausCount}")
print(f"Column Count     : {StyleHausColumnCount}")
print(f"Columns          : {StyleHausDf.columns}")

print("\nURBANTHREAD")
print("-" * 40)

print(f"Expected Records : {SourceConfigurations['UrbanThread']['ExpectedRecordCount']}")
print(f"Actual Records   : {UrbanThreadCount}")
print(f"Column Count     : {UrbanThreadColumnCount}")
print(f"Columns          : {UrbanThreadDf.columns}")

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 8, Finished, Available, Finished, False)


SOURCE DATA VALIDATION

STYLEHAUS
----------------------------------------
Expected Records : 100
Actual Records   : 94
Column Count     : 11
Columns          : ['sku', 'product_name', 'brand', 'category', 'description', 'price', 'color', 'size', 'material', 'in_stock', 'updated_at']

URBANTHREAD
----------------------------------------
Expected Records : 100
Actual Records   : 70
Column Count     : 11
Columns          : ['product_code', 'title', 'manufacturer', 'type', 'details', 'retail_price', 'colour', 'dimensions', 'fabric', 'availability', 'last_modified']


In [7]:
# ============================================================
# 7. DISPLAY SOURCE DATA
# ============================================================

print("\nSTYLEHAUS SAMPLE RECORDS")

display(StyleHausDf.limit(5))


print("\nURBANTHREAD SAMPLE RECORDS")

display(UrbanThreadDf.limit(5))

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 9, Finished, Available, Finished, False)


STYLEHAUS SAMPLE RECORDS


SynapseWidget(Synapse.DataFrame, 131cd9ff-b61c-4ba0-be26-4f6f2702c021)


URBANTHREAD SAMPLE RECORDS


SynapseWidget(Synapse.DataFrame, 04128044-0cf6-47bb-b47d-a1eb9134e11e)

In [8]:
# ============================================================
# 8. INGESTION SUMMARY
# ============================================================

print("\nBRONZE SOURCE READING SUMMARY")
print("=" * 60)

print(f"Batch ID            : {BatchID}")
print(f"StyleHaus Records   : {StyleHausCount}")
print(f"UrbanThread Records : {UrbanThreadCount}")
print(f"Total Records       : {StyleHausCount + UrbanThreadCount}")

print("\nSOURCE READING COMPLETED")

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 10, Finished, Available, Finished, False)


BRONZE SOURCE READING SUMMARY
Batch ID            : a079f8b0-7e6e-45c9-af63-f34a1698ca8b
StyleHaus Records   : 94
UrbanThread Records : 70
Total Records       : 164

SOURCE READING COMPLETED


In [9]:
# ============================================================
# ADD BRONZE INGESTION METADATA
# ============================================================

# StyleHaus
StyleHausBronzeDf = (
    StyleHausDf
    .withColumn(
        "SourceSystem",
        F.lit("StyleHaus")
    )
    .withColumn(
        "BatchID",
        F.lit(BatchID)
    )
    .withColumn(
        "IngestionTimestamp",
        F.lit(IngestionTimestamp.isoformat()).cast("timestamp")
    )
)

# UrbanThread
UrbanThreadBronzeDf = (
    UrbanThreadDf
    .withColumn(
        "SourceSystem",
        F.lit("UrbanThread")
    )
    .withColumn(
        "BatchID",
        F.lit(BatchID)
    )
    .withColumn(
        "IngestionTimestamp",
        F.lit(IngestionTimestamp.isoformat()).cast("timestamp")
    )
)

print("Bronze metadata added successfully.")

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 11, Finished, Available, Finished, False)

Bronze metadata added successfully.


In [10]:
# ============================================================
# ADD BRONZE INGESTION METADATA
# ============================================================

# StyleHaus
StyleHausBronzeDf = (
    StyleHausDf
    .withColumn(
        "SourceSystem",
        F.lit("StyleHaus")
    )
    .withColumn(
        "BatchID",
        F.lit(BatchID)
    )
    .withColumn(
        "IngestionTimestamp",
        F.lit(IngestionTimestamp.isoformat()).cast("timestamp")
    )
)

# UrbanThread
UrbanThreadBronzeDf = (
    UrbanThreadDf
    .withColumn(
        "SourceSystem",
        F.lit("UrbanThread")
    )
    .withColumn(
        "BatchID",
        F.lit(BatchID)
    )
    .withColumn(
        "IngestionTimestamp",
        F.lit(IngestionTimestamp.isoformat()).cast("timestamp")
    )
)

print("Bronze metadata added successfully.")

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 12, Finished, Available, Finished, False)

Bronze metadata added successfully.


In [11]:
# ============================================================
# WRITE BRONZE DELTA TABLES
# ============================================================

# ------------------------------------------------------------
# WRITE STYLEHAUS
# ------------------------------------------------------------

StyleHausTable = "Bronze_StyleHausProducts"

(
    StyleHausBronzeDf.write
    .format("delta")
    .mode("append")
    .saveAsTable(StyleHausTable)
)

print(f"SUCCESS: {StyleHausTable} created/updated")


# ------------------------------------------------------------
# WRITE URBANTHREAD
# ------------------------------------------------------------

UrbanThreadTable = "Bronze_UrbanThreadProducts"

(
    UrbanThreadBronzeDf.write
    .format("delta")
    .mode("append")
    .saveAsTable(UrbanThreadTable)
)

print(f"SUCCESS: {UrbanThreadTable} created/updated")

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 13, Finished, Available, Finished, False)

SUCCESS: Bronze_StyleHausProducts created/updated
SUCCESS: Bronze_UrbanThreadProducts created/updated


In [12]:
# ============================================================
# VALIDATE INGESTION METADATA
# ============================================================

print("STYLEHAUS METADATA")

spark.sql("""
    SELECT
        SourceSystem,
        BatchID,
        COUNT(*) AS RecordCount,
        MIN(IngestionTimestamp) AS FirstIngestion,
        MAX(IngestionTimestamp) AS LastIngestion
    FROM Bronze_StyleHausProducts
    GROUP BY SourceSystem, BatchID
""").show(truncate=False)


print("URBANTHREAD METADATA")

spark.sql("""
    SELECT
        SourceSystem,
        BatchID,
        COUNT(*) AS RecordCount,
        MIN(IngestionTimestamp) AS FirstIngestion,
        MAX(IngestionTimestamp) AS LastIngestion
    FROM Bronze_UrbanThreadProducts
    GROUP BY SourceSystem, BatchID
""").show(truncate=False)

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 14, Finished, Available, Finished, False)

STYLEHAUS METADATA
+------------+------------------------------------+-----------+-------------------------+-------------------------+
|SourceSystem|BatchID                             |RecordCount|FirstIngestion           |LastIngestion            |
+------------+------------------------------------+-----------+-------------------------+-------------------------+
|StyleHaus   |a079f8b0-7e6e-45c9-af63-f34a1698ca8b|94         |2026-11-03 07:48:39.87763|2026-11-03 07:48:39.87763|
+------------+------------------------------------+-----------+-------------------------+-------------------------+

URBANTHREAD METADATA
+------------+------------------------------------+-----------+-------------------------+-------------------------+
|SourceSystem|BatchID                             |RecordCount|FirstIngestion           |LastIngestion            |
+------------+------------------------------------+-----------+-------------------------+-------------------------+
|UrbanThread |a079f8b0-7e6e-45c

In [13]:
# ============================================================
# FINAL BRONZE INGESTION VALIDATION
# ============================================================

BronzeValidation = spark.sql("""
    SELECT
        SourceSystem,
        COUNT(*) AS TotalRecords,
        COUNT(DISTINCT BatchID) AS TotalBatches,
        COUNT(DISTINCT BatchID) AS DistinctBatchCount,
        SUM(
            CASE WHEN BatchID IS NULL THEN 1 ELSE 0 END
        ) AS NullBatchRecords,
        SUM(
            CASE WHEN SourceSystem IS NULL THEN 1 ELSE 0 END
        ) AS NullSourceRecords,
        SUM(
            CASE WHEN IngestionTimestamp IS NULL THEN 1 ELSE 0 END
        ) AS NullTimestampRecords
    FROM (
        SELECT SourceSystem, BatchID, IngestionTimestamp
        FROM Bronze_StyleHausProducts

        UNION ALL

        SELECT SourceSystem, BatchID, IngestionTimestamp
        FROM Bronze_UrbanThreadProducts
    )
    GROUP BY SourceSystem
""")

display(BronzeValidation)

StatementMeta(, ff7bbbae-4279-4654-8d5f-d3cc275667bc, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 06b03589-ff39-4a5a-9159-c6a3cc4c8225)